# Exploración de la fuente: API del Banco Mundial

El objetivo de este notebook no es hacer el ETL, sino **investigar la fuente** para entender la forma, calidad y metadatos de los indicadores económicos antes de modelarlos en nuestra base de datos.

In [ ]:
import yaml
import json
import pandas as pd
import sys

# Main path for extraction module
sys.path.append('../src/retail_platform/extraction/') 
from world_bank import WorldBankClient

In [ ]:
# 1. Read configuration from sources.yaml
with open('../config/sources.yaml', 'r') as file:
    config = yaml.safe_load(file)

wb_config = config['sources']['world_bank']
countries = wb_config['countries']
# Extraemos solo los códigos de la lista de diccionarios
indicators = [ind['code'] for ind in wb_config['indicators']] 
start_year = wb_config['start_year']
end_year = wb_config['end_year']

print(f"Configured countries: {countries}")
print(f"Total indicators: {len(indicators)}")
print(f"Year range: {start_year} - {end_year}")

Configured countries: ['PER', 'BRA', 'CHL', 'COL', 'MEX']
Total indicators: 8
Year range: 2018 - 2025


In [7]:
# 2. Instanciar el cliente y hacer la petición
client = WorldBankClient()

raw_data = client.get_indicators(
    countries=countries,
    indicators=indicators,
    start_year=start_year,
    end_year=end_year
)   

print(f"{len(raw_data)} observations retrieved from World Bank API.")

320 observations retrieved from World Bank API.


In [4]:
# 3. Ver el primer registro con formato bonito (pretty print)
# Esto nos ayuda a ver campos como 'country', 'indicator', 'value', 'date', etc.
print(json.dumps(raw_data[0], indent=2))

{
  "indicator": {
    "id": "NY.GDP.MKTP.CD",
    "value": "GDP (current US$)"
  },
  "country": {
    "id": "BR",
    "value": "Brazil"
  },
  "countryiso3code": "BRA",
  "date": "2025",
  "value": 2279920092492.13,
  "unit": "",
  "obs_status": "",
  "decimal": 0
}


In [5]:
# 4. Aplanar el JSON en un DataFrame tabular
df = pd.json_normalize(raw_data)

# Mostramos las primeras 5 filas para confirmar
df.head()

,countryiso3code,date,value,unit,obs_status,decimal,indicator.id,indicator.value,country.id,country.value
0,BRA,2025,2.279920e+12,,,0,NY.GDP.MKTP.CD,GDP (current US$),BR,Brazil
1,BRA,2024,2.185822e+12,,,0,NY.GDP.MKTP.CD,GDP (current US$),BR,Brazil
2,BRA,2023,2.191132e+12,,,0,NY.GDP.MKTP.CD,GDP (current US$),BR,Brazil
3,BRA,2022,1.951924e+12,,,0,NY.GDP.MKTP.CD,GDP (current US$),BR,Brazil
4,BRA,2021,1.670647e+12,,,0,NY.GDP.MKTP.CD,GDP (current US$),BR,Brazil


## Perfilamiento de los Indicadores
A continuación, responderemos las preguntas de negocio sobre la fuente de datos ejecutando consultas sobre nuestro DataFrame.

In [8]:
print("--- PERFILAMIENTO DE LA FUENTE ---\n")

# What indicator codes are there?
print("1. Indicator codes found:")
print(df['indicator.id'].unique(), "\n")

# What names/units do they carry? (Indicator name metadata)
print("2. Metadata (Names/Units) extracted directly from the API:")
for name in df['indicator.value'].unique():
    print(f"  - {name}")
print("\n")

# What years are present in the extracted data?
print("3. Years (Annual Frequency) present:")

anios = sorted(df['date'].unique())
print(anios, "\n")

# How many countries are there, and which ones are they?
print("4. Countries present in the data:")
paises = df['country.value'].unique()
print(f"Count: {len(paises)}. They are: {paises}\n")

# Can the value be null?
print("5. Can the value be null?")
nulos_totales = df['value'].isna().sum()
print(f"Yes, {nulos_totales} observations with null values (empty).")

# Breakdown of null values ​​by indicator for further investigation:
if nulos_totales > 0:
    print("\nBreakdown of null values by indicator:")
    print(df[df['value'].isna()]['indicator.value'].value_counts())

print("\n6. Notes and revisions:")
print("Checking if the API returned additional fields like 'obs_status' or notes:")
# The V2 API usually returns 'obs_status', let's see if it has anything written
print(df['obs_status'].value_counts(dropna=False))

--- PERFILAMIENTO DE LA FUENTE ---

1. Indicator codes found:
<StringArray>
[   'NY.GDP.MKTP.CD',    'NY.GDP.PCAP.CD',       'SP.POP.TOTL',
    'FP.CPI.TOTL.ZG',    'SL.UEM.TOTL.ZS',       'PA.NUS.FCRF',
    'IT.NET.USER.ZS', 'SP.URB.TOTL.IN.ZS']
Length: 8, dtype: str 

2. Metadata (Names/Units) extracted directly from the API:
  - GDP (current US$)
  - GDP per capita (current US$)
  - Population, total
  - Inflation, consumer prices (annual %)
  - Unemployment, total (% of total labor force) (modeled ILO estimate)
  - Official exchange rate (LCU per US$, period average)
  - Individuals using the Internet (% of population)
  - Urban population (% of total population)


3. Years (Annual Frequency) present:
['2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025'] 

4. Countries present in the data:
Count: 5. They are: <StringArray>
['Brazil', 'Chile', 'Colombia', 'Mexico', 'Peru']
Length: 5, dtype: str

5. Can the value be null?
Yes, 5 observations with null values (empty).

Brea